# Deteksi Watermark "sabila" — Inference App dari main.ipynb c59faca

Pipeline aplikasi:
1 video → sisipkan watermark "sabila" → Neural Codec CompressAI → extraction → CRC/FEC → TRUE/FALSE.

Notebook ini TIDAK melakukan training dan TIDAK melakukan retry.

Neural Codec mengikuti main.ipynb c59faca: CompressAI bmshj2018_factorized pretrained quality=5 dengan entropy-coded compress/decompress.

Checkpoint yang dimuat adalah checkpoint watermark yang menyimpan latent_embedder dan latent_extractor. Untuk kebutuhan bukti customer, UI menampilkan ASCII binary "sabila" 48 bit; model internal main.ipynb memakai payload 30 bit + CRC-6 + convolutional code 128 bit.


## CELL 1 — Setup


In [ ]:
# CELL 1 — Setup
from google.colab import drive
drive.mount("/content/drive")

# Use the Colab runtime's pre-installed NumPy.
# The application does not use NumPy's random module, so no NumPy RNG
# initialization or in-kernel NumPy reinstall is needed.
# CompressAI imports torch_geometric during package initialization.
# Colab may already contain an older PyG, so install a known compatible release
# explicitly instead of relying on a preinstalled satisfying version.
!pip install -q --upgrade --force-reinstall "torch-geometric==2.8.0.post1" "compressai==1.2.8" "gradio>=4.44,<7"
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print("Dependency siap.")


## CELL 2 — Configuration

This application follows the configuration of main.ipynb c59faca. It does not train or bootstrap the codec.


In [ ]:
# CELL 2 — Configuration (main.ipynb c59faca)
import os
import math
import json
import struct
import subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
FRAME_SIZE = (128, 128)
MAX_FRAMES = 64
TEMPORAL_SLOTS = 8
CODEC_BATCH_SIZE = 4
CORE_NEURAL_QUALITY = 5

PAYLOAD_BYTES = 6
PAYLOAD_ALPHABET = "abcdefghijklmnopqrstuvwxyz012345"
PAYLOAD_BITS_PER_CHAR = 5
PAYLOAD_BITS = PAYLOAD_BYTES * PAYLOAD_BITS_PER_CHAR

CRC_BITS = 6
CONV_CONSTRAINT_LENGTH = 7
CONV_TAIL_BITS = CONV_CONSTRAINT_LENGTH - 1
CONV_INPUT_BITS = PAYLOAD_BITS + CRC_BITS + CONV_TAIL_BITS
CONV_RATE = 3
CONV_CODE_BITS = CONV_INPUT_BITS * CONV_RATE
CODE_BITS = 128
CONV_PADDING_BITS = CODE_BITS - CONV_CODE_BITS

BITS_PER_SLOT = CODE_BITS // TEMPORAL_SLOTS
FRAME_SYMBOL_BITS = BITS_PER_SLOT + TEMPORAL_SLOTS

TARGET_TEXT = "sabila"
CONTROL_TEXT = "kontro"
PRESENCE_THRESHOLD = 0.70

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/Video_data/output/"
    "v9_2_temporal_latent_validated/models/"
    "best_v1010_stage2_faa0e6fc16.pth"
)

WORK_DIR = Path("/content/sabila_app")
WORK_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

def ascii_bits(text):
    return "".join(format(ord(ch), "08b") for ch in str(text))

TARGET_ASCII_BITS = ascii_bits(TARGET_TEXT)

assert PAYLOAD_BITS == 30
assert CODE_BITS == 128
assert CODE_BITS % TEMPORAL_SLOTS == 0
assert len(TARGET_ASCII_BITS) == 48

print("Device:", device)
print("Target:", TARGET_TEXT)
print("ASCII binary target (48 bit):", TARGET_ASCII_BITS)
print("Internal payload:", PAYLOAD_BITS, "bit")
print("Internal codeword:", CODE_BITS, "bit")
print("Neural Codec:", "CompressAI bmshj2018_factorized")
print("Quality:", CORE_NEURAL_QUALITY)
print("Presence threshold:", PRESENCE_THRESHOLD)
print("Checkpoint:", CHECKPOINT_PATH)



## CELL 3 — Video I/O (diambil dari main.ipynb c59faca)


In [ ]:
# 4. Video I/O
# Legacy sampling helpers from main.ipynb were removed because the application
# uses read_input_frames() below.
def get_video_fps(path):
    """Read the source video's video-stream FPS with FFprobe."""
    path = Path(path)

    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=avg_frame_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )

    if result.returncode != 0:
        detail = result.stderr.strip()
        raise RuntimeError(
            f"FFprobe gagal membaca FPS {path}: "
            f"{detail or 'tanpa pesan error'}"
        )

    value = result.stdout.strip()
    if not value or value in {"0/0", "N/A"}:
        raise ValueError(f"FPS video tidak tersedia: {path}")

    if "/" in value:
        numerator, denominator = value.split("/", 1)
        fps = float(numerator) / float(denominator)
    else:
        fps = float(value)

    if not math.isfinite(fps) or fps <= 0:
        raise ValueError(f"FPS video tidak valid: {value}")

    return fps

def frames_to_tensor(frames):
    return torch.from_numpy(np.asarray(frames)).permute(0, 3, 1, 2).float().to(device)

def tensor_to_frames(tensor):
    return tensor.detach().clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

def read_encoded_video(path, expected_frames=None):
    """Decode via FFmpeg CLI agar AV1 terbaca saat OpenCV tidak punya decoder AV1."""
    path = Path(path)
    height, width = FRAME_SIZE
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', str(path),
        '-an', '-vf', f'scale={width}:{height}:flags=area',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:1',
    ]
    result = subprocess.run(cmd, capture_output=True, timeout=120)
    bytes_per_frame = height * width * 3
    frame_count = len(result.stdout) // bytes_per_frame
    if result.returncode != 0 or frame_count == 0:
        detail = result.stderr.decode(errors='replace').strip()
        raise RuntimeError(f'FFmpeg gagal mendecode {path}: {detail}')
    usable = frame_count * bytes_per_frame
    frames = np.frombuffer(result.stdout[:usable], dtype=np.uint8).reshape(
        frame_count, height, width, 3
    ).astype(np.float32) / 255.0
    if expected_frames is not None:
        frames = frames[:expected_frames]
        if len(frames) < expected_frames:
            frames = np.concatenate([
                frames,
                np.repeat(frames[-1:], expected_frames - len(frames), axis=0),
            ], axis=0)
    return frames
def ffmpeg_encode(frames, output_path, fps, codec, crf):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames_u8 = np.clip(np.asarray(frames) * 255.0, 0, 255).astype(np.uint8)
    height, width = frames_u8.shape[1:3]
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{width}x{height}',
        '-r', str(fps), '-i', '-', '-an', '-c:v', codec,
    ]
    if codec == 'libaom-av1':
        cmd += ['-crf', str(crf), '-b:v', '0', '-cpu-used', '8', '-row-mt', '1']
    else:
        cmd += ['-crf', str(crf), '-preset', 'fast']
    cmd += ['-pix_fmt', 'yuv420p', str(output_path)]
    result = subprocess.run(cmd, input=frames_u8.tobytes(), capture_output=True, timeout=120)
    if result.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(result.stderr.decode(errors='replace'))
    return output_path
print('Video I/O siap, termasuk decode AV1 melalui FFmpeg CLI.')



## CELL 4 — Payload/FEC/temporal/extractor architecture (diambil dari main.ipynb c59faca)


In [ ]:
# 5. Payload convolutional FEC/CRC-6, temporal interleaving, dan extractor
CONV_GENERATORS = (0o171, 0o133, 0o165)
CONV_STATES = 1 << (CONV_CONSTRAINT_LENGTH - 1)
CONV_STATE_MASK = CONV_STATES - 1
CONV_REGISTER_MASK = (1 << CONV_CONSTRAINT_LENGTH) - 1

def _parity(value):
    return int(int(value).bit_count() & 1)

CONV_NEXT_STATE = np.zeros((CONV_STATES, 2), dtype=np.int16)
CONV_OUTPUT_BITS = np.zeros((CONV_STATES, 2, CONV_RATE), dtype=np.uint8)
for _state in range(CONV_STATES):
    for _input_bit in (0, 1):
        _register = ((_state << 1) | _input_bit) & CONV_REGISTER_MASK
        CONV_NEXT_STATE[_state, _input_bit] = _register & CONV_STATE_MASK
        CONV_OUTPUT_BITS[_state, _input_bit] = [
            _parity(_register & generator) for generator in CONV_GENERATORS
        ]
del _state, _input_bit, _register

def text_to_payload(text):
    text = str(text).lower()
    if len(text) != PAYLOAD_BYTES:
        raise ValueError(f'Teks harus tepat {PAYLOAD_BYTES} karakter')
    invalid = sorted(set(text) - set(PAYLOAD_ALPHABET))
    if invalid:
        raise ValueError(
            f'Karakter tidak didukung: {invalid}. Gunakan a-z atau 0-5.'
        )
    symbols = np.asarray(
        [PAYLOAD_ALPHABET.index(character) for character in text], dtype=np.uint8
    )
    bits = np.asarray([
        (int(symbol) >> shift) & 1
        for symbol in symbols for shift in range(PAYLOAD_BITS_PER_CHAR - 1, -1, -1)
    ], dtype=np.uint8)
    return torch.tensor(bits, dtype=torch.float32, device=device).unsqueeze(0)

def payload_to_text(payload_bits):
    bits = np.asarray(payload_bits, dtype=np.uint8).reshape(-1)[:PAYLOAD_BITS]
    symbols = bits.reshape(PAYLOAD_BYTES, PAYLOAD_BITS_PER_CHAR)
    weights = (1 << np.arange(PAYLOAD_BITS_PER_CHAR - 1, -1, -1)).astype(np.uint8)
    indices = symbols @ weights
    return ''.join(PAYLOAD_ALPHABET[int(index)] for index in indices)

def crc6(payload_bits):
    """CRC-6/CDMA2000-A atas 30 bit payload (polynomial 0x27)."""
    checksum = 0
    for payload_bit in np.asarray(payload_bits, dtype=np.uint8).reshape(-1):
        feedback = ((checksum >> 5) & 1) ^ int(payload_bit)
        checksum = (checksum << 1) & 0x3F
        if feedback:
            checksum ^= 0x27
    return np.asarray([
        (checksum >> shift) & 1 for shift in range(CRC_BITS - 1, -1, -1)
    ], dtype=np.uint8)

def convolutional_encode(message_bits):
    message = np.asarray(message_bits, dtype=np.uint8).reshape(-1)
    if message.size != PAYLOAD_BITS + CRC_BITS:
        raise ValueError('Convolutional encoder memerlukan 36 message bit.')
    terminated = np.concatenate([
        message, np.zeros(CONV_TAIL_BITS, dtype=np.uint8)
    ])
    state = 0
    encoded = np.zeros(CODE_BITS, dtype=np.uint8)
    cursor = 0
    for input_bit in terminated:
        bit = int(input_bit)
        encoded[cursor:cursor + CONV_RATE] = CONV_OUTPUT_BITS[state, bit]
        state = int(CONV_NEXT_STATE[state, bit])
        cursor += CONV_RATE
    if state != 0 or cursor != CONV_CODE_BITS:
        raise RuntimeError('Terminasi convolutional code gagal.')
    return encoded

def payload_to_codeword(payload):
    payload_np = payload.detach().round().to(torch.uint8).cpu().numpy()
    rows = []
    for row in payload_np:
        payload_bits = row[:PAYLOAD_BITS].astype(np.uint8)
        rows.append(convolutional_encode(np.concatenate([
            payload_bits, crc6(payload_bits)
        ])))
    return torch.tensor(np.stack(rows), dtype=torch.float32, device=device)

TARGET_PAYLOAD = text_to_payload(TARGET_TEXT)

TARGET_CODEWORD_BITS = (
    payload_to_codeword(TARGET_PAYLOAD)[0]
    .detach()
    .round()
    .to(torch.uint8)
    .cpu()
    .numpy()
)

def viterbi_decode_soft(codeword_probabilities):
    probabilities = np.clip(
        np.asarray(codeword_probabilities, dtype=np.float64).reshape(-1)[:CONV_CODE_BITS]
        .reshape(-1, CONV_RATE),
        1e-6, 1.0 - 1e-6,
    )
    if probabilities.shape[0] != CONV_INPUT_BITS:
        raise ValueError(f'Viterbi memerlukan {CODE_BITS} code probabilities.')

    metrics = np.full(CONV_STATES, np.inf, dtype=np.float64)
    metrics[0] = 0.0
    parents = np.full(
        (CONV_INPUT_BITS, CONV_STATES), -1, dtype=np.int16
    )
    next_states = np.arange(CONV_STATES, dtype=np.int16)
    input_bits = (next_states & 1).astype(np.uint8)
    predecessor_zero = next_states >> 1
    predecessor_one = predecessor_zero | (CONV_STATES >> 1)

    for time_index, symbol_probability in enumerate(probabilities):
        log_probability = np.log(symbol_probability)
        log_inverse = np.log1p(-symbol_probability)
        branch_cost = -np.sum(
            CONV_OUTPUT_BITS * log_probability.reshape(1, 1, CONV_RATE) +
            (1 - CONV_OUTPUT_BITS) * log_inverse.reshape(1, 1, CONV_RATE),
            axis=2,
        )
        candidate_zero = (
            metrics[predecessor_zero] +
            branch_cost[predecessor_zero, input_bits]
        )
        candidate_one = (
            metrics[predecessor_one] +
            branch_cost[predecessor_one, input_bits]
        )
        choose_one = candidate_one < candidate_zero
        metrics = np.where(choose_one, candidate_one, candidate_zero)
        parents[time_index] = np.where(
            choose_one, predecessor_one, predecessor_zero
        )

    # Encoder selalu diterminasi dengan enam bit nol, jadi final state harus nol.
    state = 0
    decoded = np.empty(CONV_INPUT_BITS, dtype=np.uint8)
    for time_index in range(CONV_INPUT_BITS - 1, -1, -1):
        decoded[time_index] = state & 1
        state = int(parents[time_index, state])
    return decoded

def decode_codeword_soft(codeword_probabilities):
    probabilities = np.asarray(
        codeword_probabilities, dtype=np.float64
    ).reshape(-1)[:CODE_BITS]
    decoded = viterbi_decode_soft(probabilities)
    payload_bits = decoded[:PAYLOAD_BITS]
    checksum_bits = decoded[PAYLOAD_BITS:PAYLOAD_BITS + CRC_BITS]
    tail_bits = decoded[PAYLOAD_BITS + CRC_BITS:]
    crc_valid = bool(
        not np.any(tail_bits) and np.array_equal(checksum_bits, crc6(payload_bits))
    )
    decoded_message = decoded[:PAYLOAD_BITS + CRC_BITS]
    reconstructed_codeword = convolutional_encode(decoded_message)
    hard_codeword = (probabilities >= 0.5).astype(np.uint8)
    corrected_bits = int(np.sum(reconstructed_codeword != hard_codeword))
    return {
        'ecc_success': crc_valid,
        'crc_valid': crc_valid,
        'payload_bits': payload_bits,
        'raw_payload_bits': payload_bits.copy(),
        'decoded_text': (
            payload_to_text(payload_bits)
            if crc_valid else '<invalid-crc>'
        ),
        'raw_text': payload_to_text(payload_bits),
        'corrected_symbols': corrected_bits if crc_valid else np.nan,
        'soft_flips': corrected_bits if crc_valid else 0,
    }

def decode_codeword(codeword_bits):
    bits = np.asarray(codeword_bits, dtype=np.uint8).reshape(-1)[:CODE_BITS]
    probabilities = np.where(bits > 0, 1.0 - 1e-4, 1e-4)
    return decode_codeword_soft(probabilities)

def temporal_frame_targets(codeword, frames_per_payload):
    """Ubah [B,128] menjadi [B*T,16 bit + 8 slot-ID]."""
    batch = codeword.shape[0]
    slots = torch.arange(frames_per_payload, device=codeword.device) % TEMPORAL_SLOTS
    code_segments = codeword.view(batch, TEMPORAL_SLOTS, BITS_PER_SLOT)
    segments = code_segments[:, slots, :]
    slot_targets = slots.unsqueeze(0).expand(batch, -1)
    slot_one_hot = F.one_hot(slot_targets, TEMPORAL_SLOTS).float()
    symbols = torch.cat([segments, slot_one_hot], dim=-1)
    return (
        symbols.reshape(batch * frames_per_payload, FRAME_SYMBOL_BITS),
        segments.reshape(batch * frames_per_payload, BITS_PER_SLOT),
        slot_targets.reshape(batch * frames_per_payload),
    )

def infer_temporal_slots(slot_logits):
    """Infer phase siklik global; codec menjaga urutan frame meski kualitas turun."""
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    frame_indices = torch.arange(slot_logits.shape[0], device=slot_logits.device)
    offset_scores = []
    for offset in range(TEMPORAL_SLOTS):
        expected = (frame_indices + offset) % TEMPORAL_SLOTS
        offset_scores.append(
            torch.log(slot_probabilities[frame_indices, expected].clamp_min(1e-8)).mean()
        )
    inferred_offset = int(torch.stack(offset_scores).argmax().item())
    aligned_slots = (frame_indices + inferred_offset) % TEMPORAL_SLOTS
    confidence = float(
        slot_probabilities[frame_indices, aligned_slots].mean().item()
    )
    return aligned_slots, inferred_offset, confidence, slot_probabilities

def aggregate_temporal_predictions(segment_logits, slot_logits, presence_logits):
    """Rakit codeword dengan CRC-aware cyclic phase search.

    Perubahan V10.3:
    - semua 8 kemungkinan phase temporal diuji;
    - frame diberi bobot berdasarkan slot-confidence DAN presence;
    - logits (bukan probability) yang dirata-ratakan agar evidence lemah tetapi
      konsisten tidak hilang;
    - bila ada kandidat dengan CRC valid, kandidat itu diprioritaskan.

    Ini penting pada H.264/H.265 CRF tinggi karena classifier slot dapat memiliki
    confidence rendah walaupun urutan frame codec sebenarnya tetap terjaga.
    """
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    presence_probabilities = torch.sigmoid(presence_logits).reshape(-1)
    frame_indices = torch.arange(segment_logits.shape[0], device=segment_logits.device)

    candidates = []
    for offset in range(TEMPORAL_SLOTS):
        aligned_slots = (frame_indices + offset) % TEMPORAL_SLOTS
        aligned_slot_prob = slot_probabilities[frame_indices, aligned_slots]

        # Jangan biarkan satu frame ber-confidence sangat kecil menguasai rata-rata.
        frame_weights = (
            aligned_slot_prob.clamp_min(0.05) *
            presence_probabilities.clamp_min(0.10)
        ).to(segment_logits.dtype)

        assignment = F.one_hot(aligned_slots, TEMPORAL_SLOTS).to(
            dtype=segment_logits.dtype
        )
        weighted_assignment = assignment * frame_weights.unsqueeze(1)
        slot_mass = weighted_assignment.sum(dim=0).clamp_min(1e-6)

        assembled_logits = (
            weighted_assignment.transpose(0, 1) @ segment_logits
        ) / slot_mass.unsqueeze(1)

        code_probabilities = torch.sigmoid(assembled_logits).reshape(-1)
        code_bits = (code_probabilities >= 0.5).to(torch.uint8)

        # CRC+tail dari convolutional decoder memberikan sinyal phase yang jauh
        # lebih kuat daripada slot classifier sendirian.
        decoded = decode_codeword_soft(code_probabilities.detach().cpu().numpy())
        phase_log_likelihood = torch.log(
            aligned_slot_prob.clamp_min(1e-8)
        ).mean().item()
        bit_confidence = (
            (code_probabilities - 0.5).abs() * 2.0
        ).mean().item()

        candidates.append({
            'offset': offset,
            'aligned_slots': aligned_slots,
            'code_probabilities': code_probabilities,
            'code_bits': code_bits,
            'crc_valid': bool(decoded['crc_valid']),
            'phase_log_likelihood': float(phase_log_likelihood),
            'bit_confidence': float(bit_confidence),
            'slot_confidence': float(aligned_slot_prob.mean().item()),
        })

    crc_candidates = [c for c in candidates if c['crc_valid']]
    if crc_candidates:
        # CRC valid menjadi prioritas utama; jika lebih dari satu kandidat valid,
        # prioritaskan likelihood slot; confidence bit sebagai tie-break.
        best = max(
            crc_candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )
    else:
        # Fallback aman ketika seluruh phase gagal CRC.
        best = max(
            candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )

    slot_coverage = int(torch.unique(best['aligned_slots']).numel())
    presence = float(presence_probabilities.mean().item())

    return (
        best['code_bits'].cpu().numpy(),
        best['code_probabilities'].detach().cpu().numpy(),
        presence,
        slot_coverage,
        best['slot_confidence'],
    )

def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1

class TemporalCarrierEmbedder(nn.Module):
    def __init__(
        self, latent_channels, latent_shape, symbol_bits=FRAME_SYMBOL_BITS,
        strength=1.20, carrier_scale=0.35,
    ):
        super().__init__()
        self.latent_channels = latent_channels
        self.latent_shape = tuple(latent_shape)
        self.symbol_bits = symbol_bits
        self.strength = strength
        self.carrier_scale = carrier_scale
        carriers = torch.randn(symbol_bits, latent_channels, *self.latent_shape)
        self.carriers = nn.Parameter(carriers)
        self.refine_gain = nn.Parameter(torch.tensor(0.10))
        self.refine = nn.Sequential(
            nn.Conv2d(latent_channels * 2, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
        )

    def normalized_carriers(self):
        flat = self.carriers.flatten(1)
        scale = math.sqrt(flat.shape[1]) / flat.norm(dim=1, keepdim=True).clamp_min(1e-6)
        return (flat * scale).view_as(self.carriers)

    def forward(self, latent, frame_symbols):
        segment_symbols = frame_symbols[:, :BITS_PER_SLOT] * 2.0 - 1.0
        slot_one_hot = frame_symbols[:, BITS_PER_SLOT:]
        slot_symbols = slot_one_hot - (1.0 / TEMPORAL_SLOTS)
        centered_symbols = torch.cat([segment_symbols, slot_symbols], dim=1)
        carriers = self.normalized_carriers()
        carrier_field = torch.einsum('bs,schw->bchw', centered_symbols, carriers)
        carrier_field = carrier_field * (self.carrier_scale / math.sqrt(FRAME_SYMBOL_BITS))
        refinement = self.refine(torch.cat([latent, carrier_field], dim=1))
        gain = torch.clamp(self.refine_gain, 0.0, 0.50)
        delta = torch.tanh(carrier_field + gain * refinement)
        return latent + self.strength * delta, delta

class TemporalLatentExtractor(nn.Module):
    def __init__(self, latent_channels):
        super().__init__()
        hidden = max(192, latent_channels)
        self.features = nn.Sequential(
            nn.Conv2d(latent_channels, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.shared = nn.Sequential(
            nn.Flatten(), nn.Linear(hidden * 4 * 4, 512), nn.SiLU(), nn.Dropout(0.02)
        )
        self.segment_head = nn.Linear(512, BITS_PER_SLOT)
        self.slot_head = nn.Linear(512, TEMPORAL_SLOTS)
        self.presence_head = nn.Linear(512, 1)

    def encode_features(self, latent):
        return self.shared(self.features(latent))

    def classify(self, hidden):
        return (
            self.segment_head(hidden), self.slot_head(hidden),
            self.presence_head(hidden).squeeze(1),
        )

    def forward(self, latent):
        return self.classify(self.encode_features(latent))

target_payload = text_to_payload(TARGET_TEXT)
target_codeword = payload_to_codeword(target_payload)
assert decode_codeword(target_codeword.cpu().numpy())['decoded_text'] == TARGET_TEXT
_symbols, _segments, _slots = temporal_frame_targets(target_codeword, TEMPORAL_SLOTS)
_perfect_segment_logits = (_segments * 2.0 - 1.0) * 20.0
_perfect_slot_logits = F.one_hot(_slots, TEMPORAL_SLOTS).float() * 20.0
_assembled, _, _, _coverage, _ = aggregate_temporal_predictions(
    _perfect_segment_logits, _perfect_slot_logits,
    torch.full((TEMPORAL_SLOTS,), 20.0, device=device),
)
assert np.array_equal(
    _assembled, target_codeword.cpu().numpy().reshape(-1).astype(np.uint8)
)
assert _coverage == TEMPORAL_SLOTS
assert decode_codeword(_assembled)['decoded_text'] == TARGET_TEXT
del _symbols, _segments, _slots, _perfect_segment_logits, _perfect_slot_logits, _assembled
print(
    f'Payload {PAYLOAD_BITS} bit → convolutional FEC/CRC-6 {CODE_BITS} bit → '
    f'{TEMPORAL_SLOTS} slot × {BITS_PER_SLOT} bit.'
)


def aggregate_training_logits(segment_logits, payload_count, frames_per_payload):
    """Average corresponding slots across cycles; retain differentiability."""
    if frames_per_payload % TEMPORAL_SLOTS:
        raise ValueError('Clip harus merupakan kelipatan jumlah slot.')
    return segment_logits.reshape(
        payload_count, frames_per_payload // TEMPORAL_SLOTS,
        TEMPORAL_SLOTS, BITS_PER_SLOT,
    ).mean(dim=1).reshape(payload_count, CODE_BITS)



## CELL 5 — CompressAI Neural Codec + entropy-coded watermark bitstream (diambil dari main.ipynb c59faca)


In [ ]:
from compressai.zoo import bmshj2018_factorized

_neural_models = {}

def get_neural_model(quality):
    quality = int(quality)
    if quality not in _neural_models:
        model = bmshj2018_factorized(
            quality=quality, pretrained=True
        ).to(device).eval()
        model.update(force=True)
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        _neural_models[quality] = model
    return _neural_models[quality]

CORE_CODEC = get_neural_model(CORE_NEURAL_QUALITY)
with torch.inference_mode():
    probe_latent = CORE_CODEC.g_a(torch.zeros(1, 3, *FRAME_SIZE, device=device))
LATENT_CHANNELS = int(probe_latent.shape[1])
LATENT_SHAPE = tuple(map(int, probe_latent.shape[-2:]))

latent_embedder = TemporalCarrierEmbedder(LATENT_CHANNELS, LATENT_SHAPE).to(device)
latent_extractor = TemporalLatentExtractor(LATENT_CHANNELS).to(device)

def latent_watermark_encode(frames, output_path, fps, payload):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    codeword = payload_to_codeword(payload)
    frame_symbols, _, _ = temporal_frame_targets(codeword, len(frames))
    entries = []
    reconstructed = []
    with torch.inference_mode():
        for start in range(0, len(frames), CODEC_BATCH_SIZE):
            x = frames_to_tensor(frames[start:start + CODEC_BATCH_SIZE])
            symbols = frame_symbols[start:start + x.shape[0]]
            latent = CORE_CODEC.g_a(x)
            watermarked_latent, _ = latent_embedder(latent, symbols)
            strings = CORE_CODEC.entropy_bottleneck.compress(watermarked_latent)
            shape = tuple(map(int, watermarked_latent.shape[-2:]))
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            reconstructed.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
            entries.extend((shape, stream) for stream in strings)
    header = json.dumps({
        'codec': 'bmshj2018-factorized-latent-watermark',
        'quality': CORE_NEURAL_QUALITY, 'fps': float(fps),
        'frames': len(entries), 'height': int(frames.shape[1]),
        'width': int(frames.shape[2]), 'code_bits': CODE_BITS,
        'temporal_slots': TEMPORAL_SLOTS, 'bits_per_slot': BITS_PER_SLOT,
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'LWM1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, stream in entries:
            handle.write(struct.pack('<II', shape[0], shape[1]))
            handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return np.stack(reconstructed), output_path

def latent_watermark_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'LWM1':
            raise ValueError('Bukan container latent watermark LWM1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        entries = []
        for _ in range(header['frames']):
            h, w = struct.unpack('<II', handle.read(8))
            size = struct.unpack('<I', handle.read(4))[0]
            entries.append(((h, w), handle.read(size)))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), CODEC_BATCH_SIZE):
            batch = entries[start:start + CODEC_BATCH_SIZE]
            shape = batch[0][0]
            strings = [entry[1] for entry in batch]
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            frames.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
    return np.stack(frames), header



## CELL 6 — Load frozen watermark checkpoint

The pretrained CompressAI backbone is created in Cell 5. The frozen watermark checkpoint is loaded here. Missing checkpoints are an error; there is no training fallback.


In [ ]:
# CELL 6 — Load frozen watermark checkpoint
# TIDAK ADA TRAINING di notebook aplikasi.

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        "Checkpoint watermark tidak ditemukan:\n"
        f"  {CHECKPOINT_PATH}\n\n"
        "Notebook aplikasi tidak memiliki fallback training."
    )

saved = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

required = {
    "latent_embedder",
    "latent_extractor",
    "latent_strength",
}
missing = required - set(saved.keys())
if missing:
    raise KeyError(
        f"Checkpoint tidak kompatibel. Missing keys: {sorted(missing)}"
    )

latent_embedder.load_state_dict(
    saved["latent_embedder"],
    strict=True,
)
latent_extractor.load_state_dict(
    saved["latent_extractor"],
    strict=True,
)
latent_embedder.strength = float(
    saved["latent_strength"]
)

del saved

for model in (
    CORE_CODEC,
    latent_embedder,
    latent_extractor,
):
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)

print("Model loaded.")
print("Training: DISABLED")



## CELL 7 — Single-video pipeline


In [ ]:
# CELL 7 — Single-video pipeline

def read_input_frames(path, max_frames=MAX_FRAMES):
    """Read up to max_frames without padding short videos."""
    path = Path(path)
    height, width = FRAME_SIZE

    cmd = [
        "ffmpeg", "-hide_banner", "-loglevel", "error",
        "-i", str(path),
        "-an",
        "-vf", f"scale={width}:{height}:flags=area",
        "-frames:v", str(max_frames),
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "pipe:1",
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        timeout=120,
        check=False,
    )

    bytes_per_frame = height * width * 3
    frame_count = len(result.stdout) // bytes_per_frame

    if result.returncode != 0 or frame_count == 0:
        detail = result.stderr.decode(errors="replace").strip()
        raise RuntimeError(
            f"FFmpeg gagal membaca video {path}: "
            f"{detail or 'tanpa pesan error'}"
        )

    usable_bytes = frame_count * bytes_per_frame

    return (
        np.frombuffer(
            result.stdout[:usable_bytes],
            dtype=np.uint8,
        )
        .reshape(frame_count, height, width, 3)
        .astype(np.float32)
        / 255.0
    )


def preview_watermark(frames, payload):
    symbols, _, _ = temporal_frame_targets(
        payload_to_codeword(payload),
        len(frames),
    )

    output = []

    with torch.inference_mode():
        for start in range(
            0,
            len(frames),
            CODEC_BATCH_SIZE,
        ):
            batch = frames[
                start:start + CODEC_BATCH_SIZE
            ]

            x = frames_to_tensor(batch)
            wm_symbols = symbols[
                start:start + x.shape[0]
            ]

            latent = CORE_CODEC.g_a(x)
            wm_latent, _ = latent_embedder(
                latent,
                wm_symbols,
            )

            output.extend(
                tensor_to_frames(
                    CORE_CODEC.g_s(
                        wm_latent
                    ).clamp(0, 1)
                )
            )

    return np.asarray(output)


def extract_from_frames(frames):
    x = frames_to_tensor(frames)

    with torch.inference_mode():
        latent = CORE_CODEC.g_a(x)
        hidden = latent_extractor.encode_features(
            latent
        )

        (
            segment_logits,
            slot_logits,
            presence_logits,
        ) = latent_extractor.classify(hidden)

        (
            raw_bits,
            probabilities,
            presence,
            slot_coverage,
            slot_confidence,
        ) = aggregate_temporal_predictions(
            segment_logits,
            slot_logits,
            presence_logits,
        )

    decoded = decode_codeword_soft(
        probabilities
    )

    return {
        "raw_bits": raw_bits,
        "probabilities": probabilities,
        "presence": float(presence),
        "slot_coverage": int(slot_coverage),
        "slot_confidence": float(slot_confidence),
        "decoded": decoded,
    }


def process_video(video_path):
    if not video_path:
        raise ValueError(
            "Upload 1 video terlebih dahulu."
        )

    video_path = Path(video_path)

    all_frames = read_input_frames(
        video_path,
        max_frames=MAX_FRAMES,
    )

    available = len(all_frames)

    # c59faca short-clip policy:
    # use at most 64 frames, rounded DOWN to a complete
    # temporal cycle. Never pad.
    used = (
        min(available, MAX_FRAMES)
        // TEMPORAL_SLOTS
    ) * TEMPORAL_SLOTS

    if used < TEMPORAL_SLOTS:
        raise ValueError(
            f"Video hanya memiliki {available} frame. "
            f"Minimal {TEMPORAL_SLOTS} frame diperlukan."
        )

    frames = all_frames[:used]
    fps = get_video_fps(video_path)
    payload = text_to_payload(TARGET_TEXT)

    watermarked_preview = preview_watermark(
        frames,
        payload,
    )

    bitstream_path = (
        WORK_DIR
        / f"{video_path.stem}_sabila.lwm"
    )

    # Actual entropy-coded Neural Codec path.
    _, bitstream_path = latent_watermark_encode(
        frames,
        bitstream_path,
        fps,
        payload,
    )

    # Decode directly from LWM1 bitstream.
    decoded_frames, header = (
        latent_watermark_decode(
            bitstream_path
        )
    )

    if len(decoded_frames) != used:
        raise RuntimeError(
            f"Jumlah frame berubah: "
            f"input={used}, output={len(decoded_frames)}"
        )

    extraction = extract_from_frames(
        decoded_frames
    )

    decoded = extraction["decoded"]
    extracted_code = extraction["raw_bits"][
        :CODE_BITS
    ]

    target_code = TARGET_CODEWORD_BITS
    code_ber = float(
        np.mean(
            extracted_code != target_code
        )
    )

    target_internal = (
        TARGET_PAYLOAD[0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    recovered_internal = np.asarray(
        decoded["payload_bits"],
        dtype=np.uint8,
    )

    payload_ber = float(
        np.mean(
            recovered_internal
            != target_internal
        )
    )

    recovered_text = decoded["decoded_text"]

    payload_matches = bool(
        decoded["crc_valid"]
        and recovered_text == TARGET_TEXT
    )

    detected = bool(
        extraction["presence"]
        >= PRESENCE_THRESHOLD
        and decoded["crc_valid"]
        and payload_matches
    )

    return {
        "video_path": str(video_path),
        "target_text": TARGET_TEXT,
        "extracted_text": recovered_text,
        "target_ascii_bits": TARGET_ASCII_BITS,
        "extracted_ascii_bits": (
            ascii_bits(recovered_text)
            if recovered_text != "<invalid-crc>"
            else "<invalid-crc>"
        ),
        "target_internal_bits": "".join(
            map(str, target_internal)
        ),
        "extracted_internal_bits": "".join(
            map(str, recovered_internal)
        ),
        "target_code_bits": "".join(
            map(str, target_code)
        ),
        "extracted_code_bits": "".join(
            map(str, extracted_code)
        ),
        "code_ber": code_ber,
        "payload_ber": payload_ber,
        "payload_accuracy": 1.0 - payload_ber,
        "presence": extraction["presence"],
        "presence_threshold": PRESENCE_THRESHOLD,
        "slot_coverage": extraction["slot_coverage"],
        "slot_confidence": extraction["slot_confidence"],
        "crc_valid": bool(decoded["crc_valid"]),
        "payload_matches_target": payload_matches,
        "detected": detected,
        "status": (
            "TERDETEKSI"
            if detected
            else "TIDAK TERDETEKSI"
        ),
        "available_frames": available,
        "used_frames": used,
        "dropped_tail_frames": available - used,
        "fps": fps,
        "watermarked_preview": watermarked_preview,
        "compressed_frames": decoded_frames,
        "bitstream_path": str(bitstream_path),
        "bitstream_bytes": Path(
            bitstream_path
        ).stat().st_size,
        "header": header,
    }

print("Single-video pipeline siap.")



In [ ]:
# CELL 7b — Preview writer
import subprocess
import html

def write_preview_mp4(frames, path, fps):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    frames_u8 = np.clip(
        np.asarray(frames) * 255.0,
        0,
        255,
    ).astype(np.uint8)

    h, w = frames_u8.shape[1:3]
    cmd = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "-s", f"{w}x{h}",
        "-r", str(float(fps)),
        "-i", "-",
        "-an",
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        str(path),
    ]

    result = subprocess.run(
        cmd,
        input=frames_u8.tobytes(),
        capture_output=True,
        timeout=120,
        check=False,
    )

    if (
        result.returncode != 0
        or not path.is_file()
        or path.stat().st_size == 0
    ):
        detail = result.stderr.decode(errors="replace").strip()
        raise RuntimeError(
            f"Preview MP4 gagal: {detail or 'tanpa pesan error'}"
        )

    return str(path)



## CELL 8 — Optional sanity check


In [ ]:
# CELL 8 — Optional sanity check

def run_sanity_check(video_path):
    if not video_path:
        raise ValueError(
            "Masukkan path video."
        )

    source = read_input_frames(
        video_path,
        max_frames=MAX_FRAMES,
    )

    used = (
        min(len(source), MAX_FRAMES)
        // TEMPORAL_SLOTS
    ) * TEMPORAL_SLOTS

    if used < TEMPORAL_SLOTS:
        raise ValueError(
            "Frame tidak cukup."
        )

    source = source[:used]
    fps = get_video_fps(video_path)

    def plain_neural_roundtrip(frames):
        output = []

        with torch.inference_mode():
            for start in range(
                0,
                len(frames),
                CODEC_BATCH_SIZE,
            ):
                x = frames_to_tensor(
                    frames[
                        start:start + CODEC_BATCH_SIZE
                    ]
                )

                latent = CORE_CODEC.g_a(x)

                streams = (
                    CORE_CODEC
                    .entropy_bottleneck
                    .compress(latent)
                )

                shape = tuple(
                    map(
                        int,
                        latent.shape[-2:],
                    )
                )

                latent_hat = (
                    CORE_CODEC
                    .entropy_bottleneck
                    .decompress(
                        streams,
                        shape,
                    )
                )

                output.extend(
                    tensor_to_frames(
                        CORE_CODEC.g_s(
                            latent_hat
                        ).clamp(0, 1)
                    )
                )

        return np.asarray(output)

    negative = extract_from_frames(
        plain_neural_roundtrip(source)
    )

    qwerty_payload = text_to_payload(
        "qwerty"
    )

    qwerty_path = (
        WORK_DIR
        / "sanity_qwerty.lwm"
    )

    _, qwerty_path = latent_watermark_encode(
        source,
        qwerty_path,
        fps,
        qwerty_payload,
    )

    qwerty_frames, _ = latent_watermark_decode(
        qwerty_path
    )

    qwerty = extract_from_frames(
        qwerty_frames
    )

    target = process_video(
        video_path
    )

    print(
        "A. tanpa watermark ->",
        negative["decoded"]["decoded_text"],
        f"| presence={negative['presence']:.4f}",
        f"| crc={negative['decoded']['crc_valid']}",
    )

    print(
        "B. qwerty ->",
        qwerty["decoded"]["decoded_text"],
        f"| presence={qwerty['presence']:.4f}",
        f"| crc={qwerty['decoded']['crc_valid']}",
    )

    print(
        "C. sabila ->",
        target["extracted_text"],
        f"| presence={target['presence']:.4f}",
        f"| crc={target['crc_valid']}",
    )

    false_positive = (
        negative["decoded"]["decoded_text"]
        == TARGET_TEXT
        or qwerty["decoded"]["decoded_text"]
        == TARGET_TEXT
    )

    if false_positive:
        print(
            "VERDICT: FAIL — kontrol menghasilkan sabila."
        )
    elif target["detected"]:
        print(
            "VERDICT: PASS — target terdeteksi "
            "dan kontrol tidak menghasilkan target."
        )
    else:
        print(
            "VERDICT: target belum terdeteksi."
        )



## CELL 9 — Simple UI


In [ ]:
# CELL 9 — Simple UI
import html
import gradio as gr

def status_html(text, ok):
    background = "#1b8a3a" if ok else "#c62828"
    return (
        "<div style='padding:14px;border-radius:8px;"
        f"background:{background};color:white;"
        "font-size:26px;font-weight:bold;text-align:center'>"
        f"{html.escape(text)}"
        "</div>"
    )

def debug_text(result):
    return "\n".join([
        f"Frame tersedia       : {result['available_frames']}",
        f"Frame dipakai        : {result['used_frames']}",
        f"Tail dibuang         : {result['dropped_tail_frames']}",
        f"FPS                  : {result['fps']:.2f}",
        f"Presence             : {result['presence']:.4f}",
        f"Threshold            : {result['presence_threshold']:.2f}",
        f"CRC valid            : {result['crc_valid']}",
        f"Payload == target    : {result['payload_matches_target']}",
        f"Codeword BER (128)   : {result['code_ber']:.4f}",
        f"Payload BER (30)     : {result['payload_ber']:.4f}",
        f"Payload accuracy     : {result['payload_accuracy']:.2%}",
        f"Slot coverage        : {result['slot_coverage']}/{TEMPORAL_SLOTS}",
        f"Slot confidence      : {result['slot_confidence']:.4f}",
        f"Bitstream             : {Path(result['bitstream_path']).name}",
        f"Bitstream size        : {result['bitstream_bytes']:,} bytes",
        "",
        f"ASCII target (48 bit): {result['target_ascii_bits']}",
        f"ASCII recovered      : {result['extracted_ascii_bits']}",
        "",
        f"Internal target (30): {result['target_internal_bits']}",
        f"Internal recovered   : {result['extracted_internal_bits']}",
    ])

def ui_process(video_path):
    if not video_path:
        return (
            None, None,
            status_html("Upload 1 video dulu", False),
            TARGET_TEXT, "",
            TARGET_ASCII_BITS, "",
            "", "", "", "",
        )

    try:
        result = process_video(video_path)

        stem = Path(
            result["video_path"]
        ).stem

        watermarked_preview = write_preview_mp4(
            result["watermarked_preview"],
            WORK_DIR / f"{stem}_watermarked.mp4",
            result["fps"],
        )

        compressed_preview = write_preview_mp4(
            result["compressed_frames"],
            WORK_DIR / f"{stem}_neural_codec.mp4",
            result["fps"],
        )

        return (
            watermarked_preview,
            compressed_preview,
            status_html(
                result["status"],
                result["detected"],
            ),
            result["target_text"],
            result["extracted_text"],
            result["target_ascii_bits"],
            result["extracted_ascii_bits"],
            (
                f"Codeword BER (128): {result['code_ber']:.4f}\n"
                f"Payload BER (30): {result['payload_ber']:.4f}"
            ),
            (
                f"Payload accuracy: {result['payload_accuracy']:.2%}"
            ),
            (
                f"Presence: {result['presence']:.4f}\n"
                f"Threshold: {result['presence_threshold']:.2f}"
            ),
            debug_text(result),
        )

    except Exception as exc:
        return (
            None, None,
            status_html("ERROR", False),
            TARGET_TEXT, "",
            TARGET_ASCII_BITS, "",
            "", "", "",
            f"{type(exc).__name__}: {exc}",
        )


try:
    demo.close()
except Exception:
    pass

with gr.Blocks(
    title="Deteksi Watermark sabila"
) as demo:
    gr.Markdown(
        """
        # Deteksi Watermark sabila

        **1 video → watermark → Neural Codec → extraction → CRC/FEC → TRUE/FALSE**

        Tidak ada training dan tidak ada retry.
        """
    )

    with gr.Row():
        input_video = gr.Video(
            label="1. Video input / preview"
        )
        with gr.Column():
            process_button = gr.Button(
                "Proses",
                variant="primary",
            )
            gr.Markdown("### 4. Status deteksi")
            status = gr.HTML()

    with gr.Row():
        output_watermarked = gr.Video(
            label="2. Preview setelah watermark",
            interactive=False,
        )
        output_compressed = gr.Video(
            label="3. Hasil Neural Codec",
            interactive=False,
        )

    with gr.Row():
        target_text_box = gr.Textbox(
            label="5. Teks target",
            value=TARGET_TEXT,
            interactive=False,
        )
        extracted_text_box = gr.Textbox(
            label="6. Teks hasil ekstraksi",
            interactive=False,
        )

    with gr.Row():
        target_bits_box = gr.Textbox(
            label="7. Binary ASCII target (48 bit)",
            value=TARGET_ASCII_BITS,
            interactive=False,
        )
        extracted_bits_box = gr.Textbox(
            label="8. Binary ASCII hasil recovery (48 bit)",
            interactive=False,
        )

    with gr.Row():
        ber_box = gr.Textbox(
            label="9. BER",
            lines=2,
            interactive=False,
        )
        accuracy_box = gr.Textbox(
            label="10. Bit accuracy",
            lines=2,
            interactive=False,
        )
        threshold_box = gr.Textbox(
            label="11. Presence / threshold",
            lines=2,
            interactive=False,
        )

    with gr.Accordion(
        "12. Info ekstraksi / debug",
        open=True,
    ):
        debug_box = gr.Textbox(
            label="Debug",
            lines=18,
            interactive=False,
        )

    process_button.click(
        ui_process,
        inputs=input_video,
        outputs=[
            output_watermarked,
            output_compressed,
            status,
            target_text_box,
            extracted_text_box,
            target_bits_box,
            extracted_bits_box,
            ber_box,
            accuracy_box,
            threshold_box,
            debug_box,
        ],
    )

demo.launch(
    share=True,
    debug=True,
    prevent_thread_lock=True,
    allowed_paths=[str(WORK_DIR)],
)

